In [1]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt
import torch
from typing import List
import re

from ax.service.ax_client import AxClient
from ax.service.utils.report_utils import exp_to_df

import sys
import os
# issue with https://github.com/pytorch/pytorch/issues/37377
os.environ["MKL_THREADING_LAYER"]="GNU"
#sys.path.append(f"{os.getcwd()}")
sys.path.append(f"..")
from util import encode_parameters
sys.path.append('../../experiment/run_files')
from run_command import run_command

%load_ext autoreload
%autoreload 2

In [3]:
%pwd
%cd ../..

/home/frischs


In [ ]:
/home/frischs/dev2/scaling-laws-ecnn/NAS/data/galaxy10_weighted_folder_2/ax_client_5.json

In [6]:
dev = "scaling-laws-ecnn/"
dev1 = "dev1/" + dev
dev2 = "dev2/" + dev
storage_NAS = "NAS/data/"
save_folder = dev2 + storage_NAS + "common_best_architecture/"

galaxy10_client = AxClient.load_from_json_file(filepath= dev2 + storage_NAS + "galaxy10_weighted_folder_2/ax_client_5.json")
cifar10_client = AxClient.load_from_json_file(filepath= dev1 + storage_NAS + "cifar10_2.2/ax_client.json")
mnist_rot_client = AxClient.load_from_json_file(filepath= dev1 + storage_NAS + "mnist_rot_2.2/ax_client.json")
choice_2_range_params = {
    "group": [1, 2, 4, 8, 16],
}

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")


/home/frischs/anaconda3/envs/scaling/lib/python3.11/site-packages/ax/core/parameter.py:511: UserWarning: `sort_values` is not specified for `ChoiceParameter` "-1_expand_ratio". Defaulting to `True` for parameters of `ParameterType` INT. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  warn(
/home/frischs/anaconda3/envs/scaling/lib/python3.11/site-packages/ax/core/parameter.py:511: UserWarning: `sort_values` is not specified for `ChoiceParameter` "-1_dropout_rate". Defaulting to `True` for parameters of `ParameterType` FLOAT. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  warn(
/home/frischs/anaconda3/envs/scaling/lib/python3.11/site-packages/ax/core/parameter.py:511: UserWarning: `sort_values` is not specified for `ChoiceParameter` "0_kernel_size". Defaulting to `True` for parameters of `ParameterType` INT. To override this behavior (or avoid this warning)

[INFO 09-12 18:18:14] ax.service.ax_client: Starting optimization with verbose logging. To disable logging, set the `verbose_logging` argument to `False`. Note that float values in the logs are rounded to 6 decimal points.
/home/frischs/anaconda3/envs/scaling/lib/python3.11/site-packages/ax/core/parameter.py:511: UserWarning: `sort_values` is not specified for `ChoiceParameter` "-1_expand_ratio". Defaulting to `True` for parameters of `ParameterType` INT. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  warn(
/home/frischs/anaconda3/envs/scaling/lib/python3.11/site-packages/ax/core/parameter.py:511: UserWarning: `sort_values` is not specified for `ChoiceParameter` "-1_dropout_rate". Defaulting to `True` for parameters of `ParameterType` FLOAT. To override this behavior (or avoid this warning), specify `sort_values` during `ChoiceParameter` construction.
  warn(
/home/frischs/anaconda3/envs/scaling/lib/python3.11/site-pack

In [8]:
pd.set_option('display.max_columns', None)
galaxy_df = exp_to_df(galaxy10_client.experiment)
galaxy_df = galaxy_df.drop_duplicates(subset=['arm_name'], keep=False)
galaxy_df.sort_values(by=["valid_acc_weighted"], ascending=False).head(5)

,trial_index,arm_name,trial_status,generation_method,gflops,model_building_time,valid_acc_weighted,is_feasible,-1_expand_ratio,-1_dropout_rate,0_reflection,0_group,0_out_channels,0_kernel_size,0_stride,1_reflection,1_group,1_num_layers,1_conv_op,1_kernel_size,1_se_ratio,1_out_channels,1_skip_op,1_stride,2_reflection,2_group,2_num_layers,2_conv_op,2_kernel_size,2_se_ratio,2_out_channels,2_skip_op,2_stride,3_reflection,3_group,3_num_layers,3_conv_op,3_kernel_size,3_se_ratio,3_out_channels,3_skip_op,3_stride,4_reflection,4_group,4_out_channels,4_kernel_size,4_stride
76,76,76_0,COMPLETED,Manual,1334.167225,22.000532,0.823074,True,4,0.3,0,0,1.0,5,1,0,0,2,mbconv,3,0.0,3.540516,conv,2,0,0,2,conv,5,0.0,2.554163,no,1,0,0,2,conv,5,0.75,2.311723,identity,2,0,0,1.0,3,1
51,51,51_0,COMPLETED,Manual,1326.450522,21.334147,0.820104,True,4,0.3,0,0,1.0,5,1,0,0,2,dconv,3,0.0,3.543818,conv,2,0,0,2,conv,5,0.0,2.571793,no,1,0,0,2,conv,5,0.75,2.274017,identity,2,0,0,1.0,3,1
91,91,91_0,COMPLETED,Manual,1350.847836,21.848252,0.820000,True,4,0.3,0,0,1.0,5,1,0,0,2,mbconv,3,0.0,3.539034,conv,2,0,0,2,conv,5,0.0,2.526691,no,1,0,0,2,conv,5,0.75,2.361749,identity,2,0,0,1.0,3,1
90,90,90_0,COMPLETED,Manual,1307.641899,21.062859,0.818140,True,4,0.3,0,0,1.0,5,1,0,0,2,mbconv,3,0.0,3.598130,conv,2,0,0,2,conv,5,0.0,2.471398,no,1,0,0,2,conv,5,0.75,2.314401,identity,2,0,0,1.0,3,1
69,69,69_0,COMPLETED,Manual,1288.963902,21.612698,0.818091,True,4,0.3,0,0,1.0,5,1,0,0,1,mbconv,3,0.0,3.504185,conv,2,0,0,2,conv,5,0.0,2.540230,no,1,0,0,2,conv,5,0.75,2.313124,identity,2,0,0,1.0,3,1


In [10]:
experiment_df = exp_to_df(mnist_rot_client.experiment)
experiment_df = experiment_df.drop_duplicates(subset=['arm_name'], keep=False)
experiment_df.sort_values(by=["valid_acc"], ascending=False).head(5)

,trial_index,arm_name,trial_status,generation_method,gflops,model_building_time,valid_acc,is_feasible,-1_expand_ratio,-1_dropout_rate,0_reflection,0_group,0_out_channels,0_kernel_size,0_stride,1_reflection,1_group,1_num_layers,1_conv_op,1_kernel_size,1_se_ratio,1_out_channels,1_skip_op,1_stride,2_reflection,2_group,2_num_layers,2_conv_op,2_kernel_size,2_se_ratio,2_out_channels,2_skip_op,2_stride,3_reflection,3_group,3_num_layers,3_conv_op,3_kernel_size,3_se_ratio,3_out_channels,3_skip_op,3_stride,4_reflection,4_group,4_out_channels,4_kernel_size,4_stride
69,69,69_0,COMPLETED,FullyBayesianMOO,6.720720,2.007167,0.9990,True,6,0.0,-1,4,2.184318,5,2,-1,4,2,mbconv,3,0.0,3.665396,conv,2,-1,4,2,mbconv,5,0.75,1.000000,identity,1,-1,4,1,conv,5,0.0,1.0,identity,2,-1,4,1.00000,5,2
68,68,68_0,COMPLETED,FullyBayesianMOO,6.720720,1.946726,0.9985,True,6,0.0,-1,4,2.234066,5,2,-1,4,2,mbconv,3,0.0,3.601205,conv,2,-1,4,2,mbconv,5,0.75,1.000000,identity,1,-1,4,1,conv,5,0.0,1.0,identity,2,-1,4,1.00000,5,2
75,75,75_0,COMPLETED,FullyBayesianMOO,3.823342,1.216081,0.9980,True,6,0.0,-1,4,4.000000,5,2,-1,4,2,mbconv,3,0.0,1.322581,conv,2,-1,4,2,mbconv,5,0.75,1.000000,conv,1,-1,4,1,conv,5,0.0,1.0,conv,2,-1,4,1.00000,5,1
66,66,66_0,COMPLETED,FullyBayesianMOO,7.942426,2.232413,0.9975,True,6,0.0,-1,4,2.273852,5,2,-1,4,2,mbconv,3,0.0,3.549868,conv,2,-1,4,2,mbconv,5,0.75,1.000000,conv,1,-1,4,2,conv,5,0.0,1.0,identity,2,-1,4,1.00000,5,2
98,98,98_0,COMPLETED,FullyBayesianMOO,23.720382,1.711195,0.9970,True,6,0.0,-1,4,2.112610,5,2,-1,4,1,dconv,3,0.0,3.506219,identity,1,-1,4,2,mbconv,5,0.00,1.175772,identity,1,-1,4,2,mbconv,5,0.0,1.0,no,2,-1,4,1.98396,5,1


In [11]:
cifar_df = exp_to_df(cifar10_client.experiment)
cifar_df = cifar_df.drop_duplicates(subset=['arm_name'], keep=False)
cifar_df.sort_values(by=["valid_acc"], ascending=False).head(5)

,trial_index,arm_name,trial_status,generation_method,gflops,model_building_time,valid_acc,is_feasible,-1_expand_ratio,-1_dropout_rate,0_reflection,0_group,0_out_channels,0_kernel_size,0_stride,1_reflection,1_group,1_num_layers,1_conv_op,1_kernel_size,1_se_ratio,1_out_channels,1_skip_op,1_stride,2_reflection,2_group,2_num_layers,2_conv_op,2_kernel_size,2_se_ratio,2_out_channels,2_skip_op,2_stride,3_reflection,3_group,3_num_layers,3_conv_op,3_kernel_size,3_se_ratio,3_out_channels,3_skip_op,3_stride,4_reflection,4_group,4_out_channels,4_kernel_size,4_stride
82,82,82_0,COMPLETED,FullyBayesianMOO,35.613574,255.032393,0.9141,True,6,0.0,0,4,1.115202,5,2,0,4,2,mbconv,3,0.75,2.823070,conv,1,-1,2,1,conv,3,0.50,3.944274,no,1,-1,1,2,mbconv,5,0.50,1.315595,no,2,-1,1,1.000000,3,2
92,92,92_0,COMPLETED,FullyBayesianMOO,36.681649,245.333690,0.9128,True,6,0.0,0,4,1.630202,5,2,0,4,1,mbconv,3,0.75,2.335684,conv,1,0,2,1,conv,3,0.25,3.855957,no,1,0,2,2,mbconv,5,0.50,1.000000,identity,2,-1,2,1.138669,5,1
88,88,88_0,COMPLETED,FullyBayesianMOO,39.999106,330.269326,0.9128,True,6,0.0,0,4,1.457720,5,2,0,4,2,mbconv,3,0.75,2.474018,conv,1,0,3,1,conv,3,0.75,3.891861,no,1,0,1,2,mbconv,5,0.50,1.178681,identity,2,-1,1,1.013203,5,1
71,71,71_0,COMPLETED,FullyBayesianMOO,31.805236,242.240107,0.9123,True,6,0.0,0,4,1.354560,5,2,0,4,1,mbconv,3,0.75,2.879285,conv,1,0,2,1,conv,3,0.50,3.616353,no,1,0,1,2,mbconv,5,0.75,1.000000,conv,2,0,0,1.000000,5,2
91,91,91_0,COMPLETED,FullyBayesianMOO,33.706483,240.559125,0.9110,True,6,0.0,0,4,1.513069,5,2,0,4,1,mbconv,3,0.75,2.608892,conv,1,0,2,1,conv,3,0.50,3.709164,no,1,0,1,2,mbconv,5,0.75,1.000000,identity,2,-1,1,1.000000,5,1


In [9]:
def optimal_experiment_trials(
        client: AxClient, 
        percentage: float = 0.02, 
        top_n: int = 10,
        optimize_for: str = "valid_acc"
    ):
    """
    Pareto optimal experiment result and top n highest valid acc 

    Parameters:
        client (AxClient): The experiment client used to retrieve data.
        percentage (float): The percentage threshold for filtering.
        top_n (int): The number of top rows to append to the final DataFrame.

    Returns:
        pd.DataFrame
    """
    # Convert the experiment to a DataFrame
    experiment_df = exp_to_df(client.experiment)
    experiment_df = experiment_df.drop_duplicates(subset=['arm_name'], keep=False)

    # Calculate the maximum valid_acc value and the threshold
    max_valid_acc = experiment_df[optimize_for].max()
    threshold = percentage * max_valid_acc

    # Get pareto optimal parameters with model predictions set to False
    pareto_optimal_parameter = client.get_pareto_optimal_parameters(use_model_predictions=False)
    pareto_optimal_trial_index = pareto_optimal_parameter.keys()
    indices_to_filter = [index for index in experiment_df["trial_index"] if index in pareto_optimal_trial_index]
    pareto_optimal_df = experiment_df[experiment_df["trial_index"].isin(indices_to_filter)]

    # Get the top n highest validation accuracy rows
    top_n_high_acc_rows = experiment_df.nlargest(top_n, optimize_for)

    # Append the top 10 rows to the final filtered DataFrame
    final_df = pd.concat([pareto_optimal_df, top_n_high_acc_rows])

    # new DataFrame containing only rows with valid_acc within the threshold and delete duplicates
    final_filtered_df = final_df[abs(final_df[optimize_for] - max_valid_acc) <= threshold]
    final_filtered_df = final_filtered_df.drop_duplicates(subset=['trial_index'], keep=False)

    return final_filtered_df


mnist_rot_final = optimal_experiment_trials(mnist_rot_client, 0.02, 10)
print(f"Len of mnist_rot_final: {len(mnist_rot_final)}")

cifar10_final = optimal_experiment_trials(cifar10_client, 0.02, 10)
print(f"Len of cifar10_final: {len(cifar10_final)}")

galaxy10_final = optimal_experiment_trials(galaxy10_client, 0.02, 10, "valid_acc_weighted")
print(f"Len of galaxy10_final: {len(galaxy10_final)}")

[INFO 09-12 18:20:36] ax.modelbridge.base: Leaving out out-of-design observations for arms: 68_0, 73_0, 49_0


/home/frischs/anaconda3/envs/scaling/lib/python3.11/site-packages/ax/modelbridge/modelbridge_utils.py:854: UserWarning: FYI: The default behavior of `get_pareto_frontier_and_configs` when `transform_outcomes_and_configs` is not specified has changed. Previously, the default was `transform_outcomes_and_configs=True`; now this argument is deprecated and behavior is as if `transform_outcomes_and_configs=False`. You did not specify `transform_outcomes_and_configs`, so this warning requires no action.
  warnings.warn(


Len of mnist_rot_final: 12


/home/frischs/anaconda3/envs/scaling/lib/python3.11/site-packages/ax/modelbridge/modelbridge_utils.py:854: UserWarning: FYI: The default behavior of `get_pareto_frontier_and_configs` when `transform_outcomes_and_configs` is not specified has changed. Previously, the default was `transform_outcomes_and_configs=True`; now this argument is deprecated and behavior is as if `transform_outcomes_and_configs=False`. You did not specify `transform_outcomes_and_configs`, so this warning requires no action.
  warnings.warn(
[WARNING 09-12 18:20:38] ax.modelbridge.modelbridge_utils: ObservationData(metric_names=['model_building_time'], means=[600.], covariance=[[nan]]) is missing the metrics {'gflops', 'valid_acc'}. Ignoring the data for the remaining metrics.
[INFO 09-12 18:20:38] ax.modelbridge.base: Leaving out out-of-design observations for arms: 97_0


Len of cifar10_final: 12
Len of galaxy10_final: 6


/home/frischs/anaconda3/envs/scaling/lib/python3.11/site-packages/ax/modelbridge/modelbridge_utils.py:854: UserWarning: FYI: The default behavior of `get_pareto_frontier_and_configs` when `transform_outcomes_and_configs` is not specified has changed. Previously, the default was `transform_outcomes_and_configs=True`; now this argument is deprecated and behavior is as if `transform_outcomes_and_configs=False`. You did not specify `transform_outcomes_and_configs`, so this warning requires no action.
  warnings.warn(


In [10]:
mnist_rot_final.to_csv(save_folder + 'mnist_rot_selected_best.csv', index=True, header=True)
cifar10_final.to_csv(save_folder + 'cifar10_selected_best.csv', index=True, header=True)
galaxy10_final.to_csv(save_folder + 'galaxy10_selected_best.csv', index=True, header=True)

In [13]:
def get_weights_for_weighted_average(
        df: pd.DataFrame, 
        valid_acc_impact: float = 0.5, 
        epsilon: float = 0.01,
        optimize_for: str = "valid_acc",  
    ):
    valid_acc_obs = df[optimize_for]
    gflops_obs = df['gflops']

    # normalize valid_acc_weights and gflops_weights to [epsilon, 1]
    valid_acc_weights = epsilon + (1 - epsilon) * (valid_acc_obs - valid_acc_obs.min()) / (valid_acc_obs.max() - valid_acc_obs.min())
    gflops_weights = epsilon + (1 - epsilon) * (gflops_obs - gflops_obs.min()) / (gflops_obs.max() - gflops_obs.min())
    #print("valid_acc_weights", valid_acc_weights)
    #print("gflops_weights", gflops_weights)
    
    weights = valid_acc_impact * valid_acc_weights + (1 - valid_acc_impact) * gflops_weights
    return weights / weights.sum()


get_weights_for_weighted_average(galaxy10_final, optimize_for="valid_acc_weighted")

34    0.003527
91    0.325796
68    0.173773
82    0.159733
97    0.213388
94    0.123783
dtype: float64

In [14]:
def summarize_optimal_architectures(
        df: pd.DataFrame, 
        valid_acc_weight: float = 0.8, 
        exclude_columns: List[str] = ['trial_index', 'arm_name', 'trial_status', 
                     'generation_method', 'model_building_time', 'is_feasible'],
        optimize_for: str = "valid_acc",
    ):

    weights = get_weights_for_weighted_average(df, valid_acc_weight, optimize_for=optimize_for)

    summary = {}
    for col in df.columns:
        if col not in exclude_columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                summary[col] = (df[col] * weights).sum()
            else:


                counts = df.groupby(col).apply(lambda x: (weights.loc[x.index]).sum())
                summary_string = ""
                for value, percentage in (counts / counts.sum() * 100).items():
                    summary_string += f"{value}={int(percentage)}%; "
                summary[col] = summary_string

    summary_df = pd.DataFrame(summary, index=[f'weighted_average'])
    return summary_df

summary_df = summarize_optimal_architectures(galaxy10_final, valid_acc_weight=0.8, optimize_for="valid_acc_weighted")  # 0.8 is the weight for valid_acc in the averaging
summary_df

,gflops,valid_acc_weighted,-1_expand_ratio,-1_dropout_rate,0_reflection,0_group,0_out_channels,0_kernel_size,0_stride,1_reflection,1_group,1_num_layers,1_conv_op,1_kernel_size,1_se_ratio,1_out_channels,1_skip_op,1_stride,2_reflection,2_group,2_num_layers,2_conv_op,2_kernel_size,2_se_ratio,2_out_channels,2_skip_op,2_stride,3_reflection,3_group,3_num_layers,3_conv_op,3_kernel_size,3_se_ratio,3_out_channels,3_skip_op,3_stride,4_reflection,4_group,4_out_channels,4_kernel_size,4_stride
weighted_average,1341.585095,0.816023,4.008888,0.299556,0.0,0.008888,1.0,5.0,1.0,0.0,0.0,2.0,dconv=19%; mbconv=80%;,3.0,0.0,3.521038,conv=99%; identity=0%;,2.0,0.0,0.0,2.0,conv=100%;,5.0,0.0,2.552879,no=100%;,1.0,0.0,0.0,2.0,conv=100%;,5.0,0.75,2.338836,identity=100%;,2.0,0.0,0.0,1.0,3.0,1.0


In [25]:
# List of DataFrames
dataframes = [mnist_rot_final, cifar10_final, galaxy10_final]  # Replace these with your actual DataFrames
dataset_names = ['mnist_rot', 'cifar10', 'galaxy10']
valid_acc_weight = 0.8

# Create a list of summary DataFrames
summary_dfs = []
for name, df in zip(dataset_names, dataframes):
    if name == "galaxy10":
        summary_dfs.append(summarize_optimal_architectures(df, valid_acc_weight=valid_acc_weight, optimize_for="valid_acc_weighted"))
    else:
        summary_dfs.append(summarize_optimal_architectures(df, valid_acc_weight=valid_acc_weight))

#summary_dfs = [summarize_optimal_architectures(df, valid_acc_weight=valid_acc_weight) for name, df in zip(dataset_names, dataframes) if name]

# Set the dataset name as the index for each summary DataFrame
for name, summary_df in zip(dataset_names, summary_dfs):
    summary_df.index = [name]

# Concatenate the summary DataFrames into one
final_summary_df = pd.concat(summary_dfs)

# Now you can print or use final_summary_df
final_summary_df

final_summary_df["valid_acc(_weighted)"] = final_summary_df["valid_acc"].combine_first(final_summary_df["valid_acc_weighted"])
final_summary_df = final_summary_df.drop(columns=["valid_acc", "valid_acc_weighted"])
# Get the last column name
last_column_name = final_summary_df.columns[-1]
# Create a list of column names with the last column in the second position
new_column_order = [final_summary_df.columns[0], last_column_name] + list(final_summary_df.columns[1:-1])
# Reorder the columns based on the new order
final_summary_df = final_summary_df[new_column_order]
final_summary_df


#final_summary_df

,gflops,valid_acc(_weighted),-1_expand_ratio,-1_dropout_rate,0_reflection,0_group,0_out_channels,0_kernel_size,0_stride,1_reflection,1_group,1_num_layers,1_conv_op,1_kernel_size,1_se_ratio,1_out_channels,1_skip_op,1_stride,2_reflection,2_group,2_num_layers,2_conv_op,2_kernel_size,2_se_ratio,2_out_channels,2_skip_op,2_stride,3_reflection,3_group,3_num_layers,3_conv_op,3_kernel_size,3_se_ratio,3_out_channels,3_skip_op,3_stride,4_reflection,4_group,4_out_channels,4_kernel_size,4_stride
mnist_rot,10.313189,0.996727,5.810005,0.000000,-0.842003,3.920021,2.277761,5.0,2.0,-0.842003,3.920021,1.694801,dconv=21%; mbconv=78%;,3.077175,0.029992,3.416244,conv=79%; identity=11%; no=9%;,1.481796,-1.000000,3.761735,1.798999,dconv=41%; mbconv=58%;,4.873424,0.662653,1.020638,conv=22%; identity=65%; no=12%;,1.001402,-1.000000,3.761735,1.313443,conv=55%; dconv=3%; mbconv=40%;,5.0,0.048846,1.207038,conv=22%; identity=51%; no=26%;,2.0,-1.000000,3.681756,1.210116,4.800051,1.801342
cifar10,36.297857,0.910632,6.000000,0.000000,0.000000,4.000000,1.510149,5.0,2.0,0.000000,4.000000,1.503390,mbconv=100%;,3.000000,0.750000,2.484065,conv=100%;,1.000000,-0.288384,2.553163,1.000000,conv=91%; dconv=8%;,3.000000,0.532853,3.830941,no=100%;,1.000000,-0.468388,1.320810,2.000000,mbconv=100%;,5.0,0.601139,1.095340,conv=19%; identity=51%; no=28%;,2.0,-0.899416,1.058785,1.016978,4.419460,1.568973
galaxy10,1341.585095,0.816023,4.008888,0.299556,0.000000,0.008888,1.000000,5.0,1.0,0.000000,0.000000,2.000000,dconv=19%; mbconv=80%;,3.000000,0.000000,3.521038,conv=99%; identity=0%;,2.000000,0.000000,0.000000,2.000000,conv=100%;,5.000000,0.000000,2.552879,no=100%;,1.000000,0.000000,0.000000,2.000000,conv=100%;,5.0,0.750000,2.338836,identity=100%;,2.0,0.000000,0.000000,1.000000,3.000000,1.000000


In [26]:
# save result_df to csv
final_summary_df.to_csv(save_folder + 'weighted_average_summaries.csv', index=True, header=True)

In [39]:
final_summary_df

,gflops,valid_acc(_weighted),-1_expand_ratio,-1_dropout_rate,0_reflection,0_group,0_out_channels,0_kernel_size,0_stride,1_reflection,1_group,1_num_layers,1_conv_op,1_kernel_size,1_se_ratio,1_out_channels,1_skip_op,1_stride,2_reflection,2_group,2_num_layers,2_conv_op,2_kernel_size,2_se_ratio,2_out_channels,2_skip_op,2_stride,3_reflection,3_group,3_num_layers,3_conv_op,3_kernel_size,3_se_ratio,3_out_channels,3_skip_op,3_stride,4_reflection,4_group,4_out_channels,4_kernel_size,4_stride
mnist_rot,10.313189,0.996727,5.810005,0.000000,-0.842003,3.920021,2.277761,5.0,2.0,-0.842003,3.920021,1.694801,dconv=21%; mbconv=78%;,3.077175,0.029992,3.416244,conv=79%; identity=11%; no=9%;,1.481796,-1.000000,3.761735,1.798999,dconv=41%; mbconv=58%;,4.873424,0.662653,1.020638,conv=22%; identity=65%; no=12%;,1.001402,-1.000000,3.761735,1.313443,conv=55%; dconv=3%; mbconv=40%;,5.0,0.048846,1.207038,conv=22%; identity=51%; no=26%;,2.0,-1.000000,3.681756,1.210116,4.800051,1.801342
cifar10,36.297857,0.910632,6.000000,0.000000,0.000000,4.000000,1.510149,5.0,2.0,0.000000,4.000000,1.503390,mbconv=100%;,3.000000,0.750000,2.484065,conv=100%;,1.000000,-0.288384,2.553163,1.000000,conv=91%; dconv=8%;,3.000000,0.532853,3.830941,no=100%;,1.000000,-0.468388,1.320810,2.000000,mbconv=100%;,5.0,0.601139,1.095340,conv=19%; identity=51%; no=28%;,2.0,-0.899416,1.058785,1.016978,4.419460,1.568973
galaxy10,1341.585095,0.816023,4.008888,0.299556,0.000000,0.008888,1.000000,5.0,1.0,0.000000,0.000000,2.000000,dconv=19%; mbconv=80%;,3.000000,0.000000,3.521038,conv=99%; identity=0%;,2.000000,0.000000,0.000000,2.000000,conv=100%;,5.000000,0.000000,2.552879,no=100%;,1.000000,0.000000,0.000000,2.000000,conv=100%;,5.0,0.750000,2.338836,identity=100%;,2.0,0.000000,0.000000,1.000000,3.000000,1.000000


In [44]:
categorical_mask = ["conv_op", "skip_op"]
round_to_float_mask = ["se_ratio", "out_channels"]
donot_round_params = ["gflops", "valid_acc(_weighted)"]
params = {
    "no_round": donot_round_params,
    "round_to_int": [],
    "round_to_float_025": [],
    "categorical": [],
    "round_to_float_01": ["-1_dropout_rate"],
    }

for param in final_summary_df.columns.tolist():
    if sum([1 for k, v in params.items() if param in v]):
        print(f"Skipping {param} because it is already assigned")   
        continue
    
    layer = param.split("_")[0]
    variable = param.split("_")[1:]
    variable = "_".join(variable)

    if variable in categorical_mask:
        params["categorical"].append(param)
    elif variable in round_to_float_mask:
        params["round_to_float_025"].append(param)
    else:
        params["round_to_int"].append(param)


def apply_transformations(df, params):
    def round_to_nearest_multiple(x, base):
        return round(x / base) * base

    def extract_max_category(probability_string):
        #print("probability_string", probability_string)
        categories = re.findall(r'(\w+)=(\d+)%', probability_string)
        if not categories:
            return None
        max_category = max(categories, key=lambda x: int(x[1]))
        return max_category[0]

    for category, columns in params.items():
        if category == 'no_round':
            continue
        elif category == 'round_to_int':
            for column in columns:
                df[column] = df[column].apply(lambda x: round(x))
        elif "round_to_float" in category:
            float_to = category.split("_")[-1]
            if float_to == "025":
                for column in columns:
                    df[column] = df[column].apply(lambda x: round_to_nearest_multiple(x, 0.25))
            elif float_to == "01":
                for column in columns:
                    df[column] = df[column].apply(lambda x: round_to_nearest_multiple(x, 0.1))
            else:
                raise ValueError(f"float_to {float_to} not supported")
        elif category == 'categorical':
            for column in columns:
                df[column] = df[column].apply(lambda x: extract_max_category(x))

    df.index = ["pure_" + index for index in df.index]
    return df

# Apply transformations
result_df = apply_transformations(final_summary_df.copy(), params)
result_df

Skipping gflops because it is already assigned
Skipping valid_acc(_weighted) because it is already assigned
Skipping -1_dropout_rate because it is already assigned


,gflops,valid_acc(_weighted),-1_expand_ratio,-1_dropout_rate,0_reflection,0_group,0_out_channels,0_kernel_size,0_stride,1_reflection,1_group,1_num_layers,1_conv_op,1_kernel_size,1_se_ratio,1_out_channels,1_skip_op,1_stride,2_reflection,2_group,2_num_layers,2_conv_op,2_kernel_size,2_se_ratio,2_out_channels,2_skip_op,2_stride,3_reflection,3_group,3_num_layers,3_conv_op,3_kernel_size,3_se_ratio,3_out_channels,3_skip_op,3_stride,4_reflection,4_group,4_out_channels,4_kernel_size,4_stride
pure_mnist_rot,10.313189,0.996727,6,0.0,-1,4,2.25,5,2,-1,4,2,mbconv,3,0.00,3.5,conv,1,-1,4,2,mbconv,5,0.75,1.00,identity,1,-1,4,1,conv,5,0.00,1.25,identity,2,-1,4,1.25,5,2
pure_cifar10,36.297857,0.910632,6,0.0,0,4,1.50,5,2,0,4,2,mbconv,3,0.75,2.5,conv,1,0,3,1,conv,3,0.50,3.75,no,1,0,1,2,mbconv,5,0.50,1.00,identity,2,-1,1,1.00,4,2
pure_galaxy10,1341.585095,0.816023,4,0.3,0,0,1.00,5,1,0,0,2,mbconv,3,0.00,3.5,conv,2,0,0,2,conv,5,0.00,2.50,no,1,0,0,2,conv,5,0.75,2.25,identity,2,0,0,1.00,3,1


In [45]:
# save result_df to csv
result_df = result_df.drop(columns=["gflops", "valid_acc(_weighted)"])
result_df.to_csv(save_folder + 'pure_strategies.csv', index=True, header=True)

# load result_df from csv
tmp = pd.read_csv(save_folder + 'pure_strategies.csv', index_col=0)
tmp


,-1_expand_ratio,-1_dropout_rate,0_reflection,0_group,0_out_channels,0_kernel_size,0_stride,1_reflection,1_group,1_num_layers,1_conv_op,1_kernel_size,1_se_ratio,1_out_channels,1_skip_op,1_stride,2_reflection,2_group,2_num_layers,2_conv_op,2_kernel_size,2_se_ratio,2_out_channels,2_skip_op,2_stride,3_reflection,3_group,3_num_layers,3_conv_op,3_kernel_size,3_se_ratio,3_out_channels,3_skip_op,3_stride,4_reflection,4_group,4_out_channels,4_kernel_size,4_stride
pure_mnist_rot,6,0.0,-1,4,2.25,5,2,-1,4,2,mbconv,3,0.00,3.5,conv,1,-1,4,2,mbconv,5,0.75,1.00,identity,1,-1,4,1,conv,5,0.00,1.25,identity,2,-1,4,1.25,5,2
pure_cifar10,6,0.0,0,4,1.50,5,2,0,4,2,mbconv,3,0.75,2.5,conv,1,0,3,1,conv,3,0.50,3.75,no,1,0,1,2,mbconv,5,0.50,1.00,identity,2,-1,1,1.00,4,2
pure_galaxy10,4,0.3,0,0,1.00,5,1,0,0,2,mbconv,3,0.00,3.5,conv,2,0,0,2,conv,5,0.00,2.50,no,1,0,0,2,conv,5,0.75,2.25,identity,2,0,0,1.00,3,1


In [34]:
global_args = ["model=eq_nasnet"]
for strategy_row in result_df.iterrows():
    strategy_name = "_".join(strategy_row[0].split('_')[1:])
    print(f"\nPure strategy: {strategy_name}")

    for dataset_row in result_df.iterrows():
        
        dataset_name = "_".join(dataset_row[0].split('_')[1:])
        print(f"Dataset: {dataset_name}")
        strategy_dict = strategy_row[1].to_dict()
        dataset_dict = dataset_row[1].to_dict()

        # we replace the (X_reflection, X_group) of strategy with the one of dataset
        for key in strategy_dict.keys():
            if '_reflection' in key or '_group' in key:
                strategy_dict[key] = dataset_dict[key]

        blocks_args = encode_parameters(strategy_dict, choice_2_range_params)
        print(f"Blocks args: {blocks_args}")

        # Create a new run
        args = [
            f"training={dataset_name}-training",
            f"wandb.tags=[eqnasnet,find_best,pure_strategy]",
            f"model.blocks_args={blocks_args}",
            f"model.dropout_rate={strategy_dict['-1_dropout_rate']}",
            f"model.eq_expand_ratio={strategy_dict['-1_expand_ratio']}",
            f"wandb.give_name=pure_strategy_{strategy_name}"
        ]

        print(f"Args: {args}")
        run_command(args, global_args, test="instantiation", path="experiment/")

        break
    break
        


Pure strategy: mnist_rot
Dataset: mnist_rot
Blocks args: ['r-1_k5_g16_o2.25_s2', 'r-1_k3_g16_o3.5_s1_n2_c-mbconv_se0.0_sk-conv', 'r-1_k5_g16_o1.0_s1_n2_c-mbconv_se0.75_sk-identity', 'r-1_k5_g16_o1.25_s2_n1_c-conv_se0.0_sk-identity', 'r-1_k5_g16_o1.25']
Args: ['training=mnist_rot-training', 'wandb.tags=[eqnasnet,find_best,pure_strategy]', "model.blocks_args=['r-1_k5_g16_o2.25_s2', 'r-1_k3_g16_o3.5_s1_n2_c-mbconv_se0.0_sk-conv', 'r-1_k5_g16_o1.0_s1_n2_c-mbconv_se0.75_sk-identity', 'r-1_k5_g16_o1.25_s2_n1_c-conv_se0.0_sk-identity', 'r-1_k5_g16_o1.25']", 'model.dropout_rate=0.0', 'model.eq_expand_ratio=6', 'wandb.give_name=pure_strategy_mnist_rot']


[2023-08-11 20:40:53,986][HYDRA] Launching 1 jobs locally
[2023-08-11 20:40:53,986][HYDRA] 	#0 : model=eq_nasnet wandb.mode=disabled training.steps_per_epoch=10 training.epochs=1 training=mnist_rot-training wandb.tags=[eqnasnet,find_best,pure_strategy] model.blocks_args=['r-1_k5_g16_o2.25_s2','r-1_k3_g16_o3.5_s1_n2_c-mbconv_se0.0_sk-conv','r-1_k5_g16_o1.0_s1_n2_c-mbconv_se0.75_sk-identity','r-1_k5_g16_o1.25_s2_n1_c-conv_se0.0_sk-identity','r-1_k5_g16_o1.25'] model.dropout_rate=0.0 model.eq_expand_ratio=6 wandb.give_name=pure_strategy_mnist_rot
Equivariant_NAS_Net
list_out_channel: [36.0, 126.0, 126.0, 157.5, 196.875]
eq_nasnetC16
Building stem
Building block: 1
Building block: 2
Building block: 3
Building head
pooling image size:  [7, 7]
{'model_building_time': 17.894424085505307, 'param_count': 3.605939, 'GFLOPs': 238.4818875}


In [17]:
columns = final_summary_df.columns.tolist()
columns.remove("gflops")
columns.remove("valid_acc")
columns
config_str = "6	0.0	0	4.0	2	5.0	2.0	0.0	4.0	1.0	mbconv,conv	3.0	0.0,0.5	1.0,2.5	conv	1.0	0.0	*	2.0	conv	3	0.25,0.5 	3.8	no	1.0	-1	*	2	mbconv	5	0.0,0.5	1.0	conv	2	-1	*	1.0	3.0,5.0	2"
config_list = []
temp_item = ""

def convert_to_number(val):
    try:
        if '.' in val:
            return float(val)
        else:
            return int(val)
    except ValueError:
        return val

config_list = []
temp_item = ""

for item in config_str.split():
    if ',' in item:
        sub_items = [convert_to_number(sub_item) for sub_item in item.split(',')]
        config_list.append(sub_items)
    else:
        config_list.append(convert_to_number(item))

        # Check if the item ends with a comma
        if item.endswith(','):
            temp_item = item[:-1]

print(config_list)
dict_config = dict(zip(columns, config_list))
dict_config

[6, 0.0, 0, 4.0, 2, 5.0, 2.0, 0.0, 4.0, 1.0, ['mbconv', 'conv'], 3.0, [0.0, 0.5], [1.0, 2.5], 'conv', 1.0, 0.0, '*', 2.0, 'conv', 3, [0.25, 0.5], 3.8, 'no', 1.0, -1, '*', 2, 'mbconv', 5, [0.0, 0.5], 1.0, 'conv', 2, -1, '*', 1.0, [3.0, 5.0], 2]
{'-1_expand_ratio': 6, '-1_dropout_rate': 0.0, '0_reflection': 0, '0_group': 4.0, '0_out_channels': 2, '0_kernel_size': 5.0, '0_stride': 2.0, '1_reflection': 0.0, '1_group': 4.0, '1_num_layers': 1.0, '1_conv_op': ['mbconv', 'conv'], '1_kernel_size': 3.0, '1_se_ratio': [0.0, 0.5], '1_out_channels': [1.0, 2.5], '1_skip_op': 'conv', '1_stride': 1.0, '2_reflection': 0.0, '2_group': '*', '2_num_layers': 2.0, '2_conv_op': 'conv', '2_kernel_size': 3, '2_se_ratio': [0.25, 0.5], '2_out_channels': 3.8, '2_skip_op': 'no', '2_stride': 1.0, '3_reflection': -1, '3_group': '*', '3_num_layers': 2, '3_conv_op': 'mbconv', '3_kernel_size': 5, '3_se_ratio': [0.0, 0.5], '3_out_channels': 1.0, '3_skip_op': 'conv', '3_stride': 2, '4_reflection': -1, '4_group': '*', '4_

In [ ]:
import random
random.seed(42)

global_args = ["model=eq_nasnet"]

groups_for_datasets = {
    "cifar10": [4, 4, 2, 1, 1], # last might also be 0
    "mnist_rot": [4, 4, 4, 4, 4],
    "galaxy10": [4, 2, 2, 2, 2], # last might also be 1
} 

choice_2_range_params = {
    "group": [1, 2, 4, 8, 16],
}

# Function to create a random configuration
def create_random_configuration(config):
    random_config = {}
    for key, value in config.items():
        if isinstance(value, list):
            random_choice = random.choice(value)
            random_config[key] = random_choice
        else:
            random_config[key] = value
            
    return random_config

def fill_group_based_on_dataset(config, dataset):
    for key, value in config.items():
        if value == "*":
            value = groups_for_datasets[dataset][int(key.split("_")[0])]
            config[key] = value
    return config

# Number of runs
num_runs = 5

# Keep track of random configurations
random_configurations = []

# Iterate for each run
for i in range(num_runs):
    while True:
        random_config = create_random_configuration(configurations)
        if random_config not in random_configurations:
            random_configurations.append(random_config)
            break

    print(f'\nRandom Configuration {i+1}: {random_config}')

    for dataset in groups_for_datasets.keys():
        config = fill_group_based_on_dataset(random_config.copy(), dataset)
        only_group_info = {k: v for k, v in config.items() if k.split("_")[1] == "group"}

        blocks_args = encode_parameters(config, choice_2_range_params)
        print(f'{i+1} for {dataset}: {blocks_args}')

        # Create a new run
        args = [
            f"training={dataset}-training",
            f"wandb.tags=[eqnas_result_test_random, {str(random_config)}]",
            f"+model={config}",
            f"model.blocks_args={blocks_args}",
        ]

        run_command(args, global_args, test="instantiation", path="../experiment/")

        """    
        # Create a new run
        run = wandb.init(
            project="nas",
            entity="eqnas",
            group=f"random_configurations_{dataset}",
            job_type="random_configurations",
            config=config,
            reinit=True,
        )

        # Run the experiment
        run_experiment(config, dataset)

        # Finish the run
        run.finish()

        # Delete the run
        del run

        # Reset the GPU
        torch.cuda.empty_cache()

        # Sleep for 5 seconds
        time.sleep(5)
        """


Random Configuration 1: {'-1_expand_ratio': 6, '-1_dropout_rate': 0.0, '0_reflection': 0, '0_group': 4.0, '0_out_channels': 2, '0_kernel_size': 5.0, '0_stride': 2.0, '1_reflection': 0.0, '1_group': 4.0, '1_num_layers': 1.0, '1_conv_op': 'mbconv', '1_kernel_size': 3.0, '1_se_ratio': 0.0, '1_out_channels': 2.5, '1_skip_op': 'conv', '1_stride': 1.0, '2_reflection': 0.0, '2_group': '*', '2_num_layers': 2.0, '2_conv_op': 'conv', '2_kernel_size': 3, '2_se_ratio': 0.25, '2_out_channels': 3.8, '2_skip_op': 'no', '2_stride': 1.0, '3_reflection': -1, '3_group': '*', '3_num_layers': 2, '3_conv_op': 'mbconv', '3_kernel_size': 5, '3_se_ratio': 0.0, '3_out_channels': 1.0, '3_skip_op': 'conv', '3_stride': 2, '4_reflection': -1, '4_group': '*', '4_out_channels': 1.0, '4_kernel_size': 3.0, '4_stride': 2}
encoded_blocks ['r0_k5.0_g16_o2_s2.0', 'r0.0_k3.0_g16_o2.5_s1.0_n1_c-mbconv_se0.0_sk-conv', 'r0.0_k3_g4_o3.8_s1.0_n2_c-conv_se0.25_sk-no', 'r-1_k5_g2_o1.0_s2_n2_c-mbconv_se0.0_sk-conv', 'r-1_k3.0_g2_o

Error: mkl-service + Intel(R) MKL: MKL_THREADING_LAYER=INTEL is incompatible with libgomp.so.1 library.
	Try to import numpy first or set the threading layer accordingly. Set MKL_SERVICE_FORCE_INTEL to force it.


encoded_blocks ['r0_k5.0_g16_o2_s2.0', 'r0.0_k3.0_g16_o2.5_s1.0_n1_c-mbconv_se0.0_sk-conv', 'r0.0_k3_g16_o3.8_s1.0_n2_c-conv_se0.25_sk-no', 'r-1_k5_g16_o1.0_s2_n2_c-mbconv_se0.0_sk-conv', 'r-1_k3.0_g16_o1.0']
1 for mnist_rot: ['r0_k5.0_g16_o2_s2.0', 'r0.0_k3.0_g16_o2.5_s1.0_n1_c-mbconv_se0.0_sk-conv', 'r0.0_k3_g16_o3.8_s1.0_n2_c-conv_se0.25_sk-no', 'r-1_k5_g16_o1.0_s2_n2_c-mbconv_se0.0_sk-conv', 'r-1_k3.0_g16_o1.0']


Error: mkl-service + Intel(R) MKL: MKL_THREADING_LAYER=INTEL is incompatible with libgomp.so.1 library.
	Try to import numpy first or set the threading layer accordingly. Set MKL_SERVICE_FORCE_INTEL to force it.


encoded_blocks ['r0_k5.0_g16_o2_s2.0', 'r0.0_k3.0_g16_o2.5_s1.0_n1_c-mbconv_se0.0_sk-conv', 'r0.0_k3_g4_o3.8_s1.0_n2_c-conv_se0.25_sk-no', 'r-1_k5_g4_o1.0_s2_n2_c-mbconv_se0.0_sk-conv', 'r-1_k3.0_g4_o1.0']
1 for galaxy10: ['r0_k5.0_g16_o2_s2.0', 'r0.0_k3.0_g16_o2.5_s1.0_n1_c-mbconv_se0.0_sk-conv', 'r0.0_k3_g4_o3.8_s1.0_n2_c-conv_se0.25_sk-no', 'r-1_k5_g4_o1.0_s2_n2_c-mbconv_se0.0_sk-conv', 'r-1_k3.0_g4_o1.0']


...

Error: mkl-service + Intel(R) MKL: MKL_THREADING_LAYER=INTEL is incompatible with libgomp.so.1 library.
	Try to import numpy first or set the threading layer accordingly. Set MKL_SERVICE_FORCE_INTEL to force it.


encoded_blocks ['r0_k5.0_g16_o2_s2.0', 'r0.0_k3.0_g16_o2.5_s1.0_n1_c-conv_se0.0_sk-conv', 'r0.0_k3_g16_o3.8_s1.0_n2_c-conv_se0.5_sk-no', 'r-1_k5_g16_o1.0_s2_n2_c-mbconv_se0.5_sk-conv', 'r-1_k3.0_g16_o1.0']
5 for mnist_rot: ['r0_k5.0_g16_o2_s2.0', 'r0.0_k3.0_g16_o2.5_s1.0_n1_c-conv_se0.0_sk-conv', 'r0.0_k3_g16_o3.8_s1.0_n2_c-conv_se0.5_sk-no', 'r-1_k5_g16_o1.0_s2_n2_c-mbconv_se0.5_sk-conv', 'r-1_k3.0_g16_o1.0']


Error: mkl-service + Intel(R) MKL: MKL_THREADING_LAYER=INTEL is incompatible with libgomp.so.1 library.
	Try to import numpy first or set the threading layer accordingly. Set MKL_SERVICE_FORCE_INTEL to force it.


encoded_blocks ['r0_k5.0_g16_o2_s2.0', 'r0.0_k3.0_g16_o2.5_s1.0_n1_c-conv_se0.0_sk-conv', 'r0.0_k3_g4_o3.8_s1.0_n2_c-conv_se0.5_sk-no', 'r-1_k5_g4_o1.0_s2_n2_c-mbconv_se0.5_sk-conv', 'r-1_k3.0_g4_o1.0']
5 for galaxy10: ['r0_k5.0_g16_o2_s2.0', 'r0.0_k3.0_g16_o2.5_s1.0_n1_c-conv_se0.0_sk-conv', 'r0.0_k3_g4_o3.8_s1.0_n2_c-conv_se0.5_sk-no', 'r-1_k5_g4_o1.0_s2_n2_c-mbconv_se0.5_sk-conv', 'r-1_k3.0_g4_o1.0']


Error: mkl-service + Intel(R) MKL: MKL_THREADING_LAYER=INTEL is incompatible with libgomp.so.1 library.
	Try to import numpy first or set the threading layer accordingly. Set MKL_SERVICE_FORCE_INTEL to force it.
